# Prompt Engineering Techniques: Reasoning Techniques

Source slide: `slides/prompt-engineering/04.2_Prompt_Engineering_Techniques.Reasoning_Techniques.md`

Each reasoning technique below uses one compact failure run and one improved run.



## Prerequisites

- Install the real GitHub Copilot SDK: `pip install github-copilot-sdk`
- Sign in to GitHub Copilot in VS Code or GitHub Codespaces
- These examples use ambient auth; do not paste a PAT into the notebook

Each technique below has a real Copilot SDK failure run, a short failure test, and an improved run that shows the fix.


In [ ]:
from copilot import CopilotClient
from copilot.session import PermissionHandler

model = "gpt-4.1"

async def ask_copilot(
    prompt: str,
    *,
    system_message: str | None = None,
    model_name: str = model,
) -> str:
    """Run a single prompt through the real GitHub Copilot SDK using ambient auth."""
    async with CopilotClient() as client:
        session_kwargs = {
            "model": model_name,
            "on_permission_request": PermissionHandler.approve_all,
        }
        if system_message:
            session_kwargs["system_message"] = {
                "mode": "append",
                "content": system_message,
            }
        async with await client.create_session(**session_kwargs) as session:
            response = await session.send_and_wait(prompt)
            return response.data.content if response and response.data else ""

def show_block(title: str, content: str) -> None:
    print(title)
    print("=" * 80)
    print(content.strip())
    print()

async def run_prompt_pair(
    *,
    technique_name: str,
    failure_prompt: str,
    improved_prompt: str,
    failure_test: str,
    fix_explanation: str,
    system_message: str | None = None,
    config_note: str | None = None,
) -> None:
    show_block("📘 Technique", technique_name)
    if config_note:
        show_block("⚙️ Configuration note", config_note)
    show_block("❌ Failure prompt", failure_prompt)
    failure_result = await ask_copilot(
        failure_prompt,
        system_message=system_message,
    )
    show_block("❌ Failure result", failure_result)
    show_block("🧪 Failure test", failure_test)
    show_block("✅ Improved prompt", improved_prompt)
    improved_result = await ask_copilot(
        improved_prompt,
        system_message=system_message,
    )
    show_block("✅ Improved result", improved_result)
    show_block("🔧 Why this fixes it", fix_explanation)

print("✅ Setup complete - real GitHub Copilot SDK with ambient auth")
print(f"📦 Using model: {model}")


## Chain-of-Thought (CoT)

**Failure mode:** CoT asks the model to surface intermediate reasoning. Without that structure, the model can jump straight to an incorrect answer on multi-step tasks.

**Failure test:** The failure prompt asks for the partner age directly, so the model may skip the age-gap calculation.

**Technique:** Ask the model to work step by step before stating the final answer.

**Improved example:** The improved prompt explicitly requests stepwise reasoning, which encourages the model to compute the age gap before answering.


In [ ]:
await run_prompt_pair(
    technique_name='Chain-of-Thought (CoT)',
    failure_prompt='When I was 3 years old, my partner was 3 times my age. Now I am 20 years old. How old is my partner?',
    improved_prompt='When I was 3 years old, my partner was 3 times my age. Now I am 20 years old. Think step by step and then state the final age on the last line.',
    failure_test='The failure prompt asks for the partner age directly, so the model may skip the age-gap calculation.',
    fix_explanation='The improved prompt explicitly requests stepwise reasoning, which encourages the model to compute the age gap before answering.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Tree-of-Thought (ToT)

**Failure mode:** ToT explores multiple candidate paths. A single-path prompt can get trapped in the first idea that appears plausible.

**Failure test:** The failure prompt asks for a meeting plan with conflicting constraints but never asks the model to compare alternatives.

**Technique:** Prompt the model to generate, compare, and discard branches before choosing one.

**Improved example:** The improved prompt asks for multiple candidate plans and an explicit comparison, which makes backtracking part of the task.


In [ ]:
await run_prompt_pair(
    technique_name='Tree-of-Thought (ToT)',
    failure_prompt='Plan a 2-day workshop agenda that covers onboarding, security, and analytics for three audiences.',
    improved_prompt='Plan a 2-day workshop agenda that covers onboarding, security, and analytics for three audiences. Generate three candidate agendas, compare their tradeoffs briefly, then pick the best one.',
    failure_test='The failure prompt asks for a meeting plan with conflicting constraints but never asks the model to compare alternatives.',
    fix_explanation='The improved prompt asks for multiple candidate plans and an explicit comparison, which makes backtracking part of the task.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Graph-of-Thought (GoT)

**Failure mode:** GoT helps when several factors influence one another. Linear prompts often miss those dependencies.

**Failure test:** The failure prompt asks for a climate plan as one flat answer, which can hide the relationships between economics, policy, and energy.

**Technique:** Ask the model to reason across connected factors instead of one straight chain.

**Improved example:** The improved prompt names the connected nodes and asks the model to explain their relationships, which encourages cross-linking instead of isolated bullets.


In [ ]:
await run_prompt_pair(
    technique_name='Graph-of-Thought (GoT)',
    failure_prompt='How do you solve climate change?',
    improved_prompt='Build a climate-change response plan by reasoning across three connected nodes: economics, renewable energy, and policy. Explain how each node affects the others before giving the final plan.',
    failure_test='The failure prompt asks for a climate plan as one flat answer, which can hide the relationships between economics, policy, and energy.',
    fix_explanation='The improved prompt names the connected nodes and asks the model to explain their relationships, which encourages cross-linking instead of isolated bullets.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Sketch-of-Thought (SoT)

**Failure mode:** SoT compresses reasoning with concise symbolic or expert shorthand. Without that compression, the model can waste tokens on verbose explanations.

**Failure test:** The failure prompt asks for a logic explanation in plain prose, which can become long and harder to inspect.

**Technique:** Ask for a concise symbolic sketch before or alongside the explanation.

**Improved example:** The improved prompt requests a compact symbolic form, which encourages shorter and more inspectable reasoning.


In [ ]:
await run_prompt_pair(
    technique_name='Sketch-of-Thought (SoT)',
    failure_prompt='Solve this logic rule and explain it in detail: If A is true then B is false, and if B is false then C is true. What happens when A is true?',
    improved_prompt='Solve this logic rule. First give a compact symbolic sketch, then a one-sentence explanation: If A is true then B is false, and if B is false then C is true. What happens when A is true?',
    failure_test='The failure prompt asks for a logic explanation in plain prose, which can become long and harder to inspect.',
    fix_explanation='The improved prompt requests a compact symbolic form, which encourages shorter and more inspectable reasoning.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Active Prompts & ReasonFlux

**Failure mode:** Active prompting improves results by adapting to new information across turns. Static prompts fail when the task needs clarification or iterative refinement.

**Failure test:** The failure prompt asks the model to tutor blindly, so it may explain the wrong concept and never adjust.

**Technique:** Prompt the model to ask clarifying questions or refine the path based on the latest response.

**Improved example:** The improved prompt authorizes one clarifying question before answering, which creates the feedback loop the technique depends on.


In [ ]:
await run_prompt_pair(
    technique_name='Active Prompts & ReasonFlux',
    failure_prompt='Teach me why my SQL query is slow.',
    improved_prompt='You are tutoring a developer on SQL performance. Ask exactly one clarifying question first if needed, then explain the most likely reason their SQL query is slow and what to inspect next.',
    failure_test='The failure prompt asks the model to tutor blindly, so it may explain the wrong concept and never adjust.',
    fix_explanation='The improved prompt authorizes one clarifying question before answering, which creates the feedback loop the technique depends on.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## ReAct (Reason & Act)

**Failure mode:** ReAct combines reasoning with external actions. If the prompt only asks for reasoning, the model may guess instead of identifying missing facts.

**Failure test:** The failure prompt asks a factual question that depends on external data, but it gives the model no permission to plan a search or retrieval step.

**Technique:** Ask the model to alternate between reasoning and information-gathering actions when facts are missing.

**Improved example:** The improved prompt explicitly asks for the missing-fact plan first, which makes the gap in evidence visible instead of hidden behind a guess.


In [ ]:
await run_prompt_pair(
    technique_name='ReAct (Reason & Act)',
    failure_prompt='How many rooms are in the hotel that hosts Mystere?',
    improved_prompt='Answer this question using a ReAct-style structure. First state the information you need, then the search actions you would take, then the final answer only if the evidence is available: How many rooms are in the hotel that hosts Mystere?',
    failure_test='The failure prompt asks a factual question that depends on external data, but it gives the model no permission to plan a search or retrieval step.',
    fix_explanation='The improved prompt explicitly asks for the missing-fact plan first, which makes the gap in evidence visible instead of hidden behind a guess.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Prompt Chaining

**Failure mode:** Prompt chaining decomposes a workflow into linked stages. One large prompt makes it hard to tell which stage failed.

**Failure test:** The failure prompt asks for legal extraction, analysis, and recommendation all at once, so the model can blur the stages together.

**Technique:** Separate extraction, analysis, and synthesis into explicit stages.

**Improved example:** The improved prompt asks the model to stage the work, which makes the pipeline easier to reason about and debug.


In [ ]:
await run_prompt_pair(
    technique_name='Prompt Chaining',
    failure_prompt='Review this clause and tell me the legal risk: “Customer data may be retained indefinitely after termination.”',
    improved_prompt='Handle this in three chained stages.\n1. Extract the operative retention clause.\n2. Explain the compliance risk in one sentence.\n3. Recommend one revision.\n\nClause: “Customer data may be retained indefinitely after termination.”',
    failure_test='The failure prompt asks for legal extraction, analysis, and recommendation all at once, so the model can blur the stages together.',
    fix_explanation='The improved prompt asks the model to stage the work, which makes the pipeline easier to reason about and debug.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Self-Consistency

**Failure mode:** Self-consistency reduces error by comparing multiple reasoning paths. A single run can still be wrong even when the prompt is reasonable.

**Failure test:** The failure prompt relies on one ambiguous classification pass, so there is no way to detect whether the answer was a fluke.

**Technique:** Rerun the task multiple times and aggregate the most consistent answer.

**Improved example:** The improved prompt asks for a label plus brief reasoning, which makes repeated runs easier to compare and vote on.


In [ ]:
await run_prompt_pair(
    technique_name='Self-Consistency',
    failure_prompt='Classify this email as IMPORTANT or ROUTINE: “Can we move tomorrow’s launch review by one hour?”',
    improved_prompt='Classify this email as IMPORTANT or ROUTINE. Return the label on the first line and one short reason on the second line: “Can we move tomorrow’s launch review by one hour?”',
    failure_test='The failure prompt relies on one ambiguous classification pass, so there is no way to detect whether the answer was a fluke.',
    fix_explanation='The improved prompt asks for a label plus brief reasoning, which makes repeated runs easier to compare and vote on.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Reflexion

**Failure mode:** Reflexion uses self-critique to catch mistakes in the first draft. Without an explicit review step, the model often stops at the first acceptable answer.

**Failure test:** The failure prompt asks for input validation code in one pass, so the model may omit important checks and never revisit the draft.

**Technique:** Add a review-and-revise loop that asks the model to inspect its own work.

**Improved example:** The improved prompt instructs the model to produce a draft, critique it, and then revise it, which creates a second pass for catching omissions.


In [ ]:
await run_prompt_pair(
    technique_name='Reflexion',
    failure_prompt='Write a Python function that validates a user signup payload.',
    improved_prompt='Write a Python function that validates a user signup payload. Then review your first draft for missing checks, and return a revised final version after the critique.',
    failure_test='The failure prompt asks for input validation code in one pass, so the model may omit important checks and never revisit the draft.',
    fix_explanation='The improved prompt instructs the model to produce a draft, critique it, and then revise it, which creates a second pass for catching omissions.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
